[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/solutions/41_segment_ids_from_lengths_solution.ipynb)

# 🟢 Solution: Segment IDs from Lengths

**Primitive: `repeat_interleave`**

**Reduction:** `output[i]` is the sequence index `j` such that `sum(lengths[:j]) <= i < sum(lengths[:j+1])`. In other words, repeat each index `j` exactly `lengths[j]` times.

An equivalent approach using `cumsum`:
1. Build a boundary tensor of zeros with length `total = sum(lengths)`
2. Place `1`s at `cumsum(lengths)[:-1]` (the start of each new segment)
3. `cumsum` the boundary tensor to get the segment IDs

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION — approach 1: repeat_interleave

def segment_ids_from_lengths(lengths: torch.Tensor) -> torch.Tensor:
    # primitive: repeat_interleave — repeats each index j exactly lengths[j] times
    return torch.repeat_interleave(torch.arange(len(lengths), device=lengths.device), lengths)


# ✅ SOLUTION — approach 2: cumsum on boundary markers (equally valid)

def segment_ids_from_lengths_v2(lengths: torch.Tensor) -> torch.Tensor:
    # primitive: cumsum — place 1s at segment boundaries, then cumsum gives the running id
    total = int(lengths.sum().item())
    if total == 0:
        return torch.zeros(0, dtype=torch.long, device=lengths.device)
    boundaries = torch.zeros(total, dtype=torch.long, device=lengths.device)
    starts = lengths.cumsum(0)[:-1]       # start index of each segment (except first)
    boundaries[starts] = 1
    return boundaries.cumsum(0)

In [ ]:
# Verify both approaches
lengths = torch.tensor([3, 1, 4, 2])
print('repeat_interleave:', segment_ids_from_lengths(lengths).tolist())
print('cumsum approach:  ', segment_ids_from_lengths_v2(lengths).tolist())
print('expected:         ', [0, 0, 0, 1, 2, 2, 2, 2, 3, 3])

# Edge cases
print('zeros [0,3,0,1]:  ', segment_ids_from_lengths(torch.tensor([0, 3, 0, 1])).tolist())
print('all-zero lengths: ', segment_ids_from_lengths(torch.tensor([0, 0, 0])).tolist())

In [ ]:
import torch, time

# ── Test 1: basic example ──────────────────────────────────────────────────
lengths = torch.tensor([3, 1, 4, 2])
result = segment_ids_from_lengths(lengths)
expected = torch.tensor([0, 0, 0, 1, 2, 2, 2, 2, 3, 3])
assert result.shape == expected.shape, f"Shape mismatch: {result.shape} vs {expected.shape}"
assert torch.equal(result, expected), f"Got {result.tolist()}, expected {expected.tolist()}"
print("Test 1 passed: basic example")

# ── Test 2: single segment ─────────────────────────────────────────────────
result = segment_ids_from_lengths(torch.tensor([5]))
expected = torch.zeros(5, dtype=torch.long)
assert torch.equal(result, expected), f"Got {result.tolist()}"
print("Test 2 passed: single segment")

# ── Test 3: zero-length segments ───────────────────────────────────────────
lengths = torch.tensor([0, 3, 0, 1])
result = segment_ids_from_lengths(lengths)
expected = torch.tensor([1, 1, 1, 3])
assert result.shape == expected.shape, f"Shape: {result.shape}"
assert torch.equal(result, expected), f"Got {result.tolist()}, expected {expected.tolist()}"
print("Test 3 passed: zero-length segments")

# ── Test 4: all-zero lengths → empty tensor ────────────────────────────────
result = segment_ids_from_lengths(torch.tensor([0, 0, 0]))
assert result.shape == (0,), f"Expected empty tensor, got shape {result.shape}"
print("Test 4 passed: all-zero lengths")

# ── Test 5: large — 1000 segments of length 1 ─────────────────────────────
lengths = torch.ones(1000, dtype=torch.long)
result = segment_ids_from_lengths(lengths)
expected = torch.arange(1000)
assert result.shape == (1000,), f"Shape: {result.shape}"
assert torch.equal(result, expected), f"Large test failed"
print("Test 5 passed: 1000 segments of length 1")

# ── Test 6: large — 10k tokens across 100 segments (timing) ───────────────
torch.manual_seed(0)
lengths = torch.randint(1, 200, (100,))
lengths = (lengths * (10000 / lengths.sum().float())).long().clamp(min=1)
t0 = time.time()
result = segment_ids_from_lengths(lengths)
elapsed = time.time() - t0
total = lengths.sum().item()
assert result.shape[0] == total, f"Total length mismatch: {result.shape[0]} vs {total}"
assert (result >= 0).all() and (result < 100).all(), "IDs out of range"
assert elapsed < 2.0, f"Too slow: {elapsed:.2f}s (expected <2s — no loops)"
print(f"Test 6 passed: 10k-token timing ({elapsed:.3f}s)")

print("\nAll tests passed!")
